In [1]:
import os
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, RemoveMessage
from pydantic import BaseModel
from langchain.tools import tool
from pydantic import BaseModel, Field
# 使用tavily作为web搜索工具
from langchain_tavily import TavilySearch, tavily_crawl
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


load_dotenv(find_dotenv())

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")



In [2]:
model = init_chat_model(
    model="qwen3.5-35b-a3b",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
    # 尝试显式关闭思考模式
    extra_body={"enable_thinking": False} 
)

In [3]:
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools = [],
    middleware=[
        SummarizationMiddleware(
            model=model,
            max_tokens_brfore_summary = 1000,
            messages_to_keep = 20,
            )
        ],
        checkpointer = checkpointer,
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

/tmp/ipykernel_2648/3059199886.py:7: DeprecationWarning: messages_to_keep is deprecated. Use keep=('messages', value) instead.
  SummarizationMiddleware(


================================== Ai Message ==================================

Your name is Bob! You told me at the very beginning of our conversation.
